# cloudflare
> The `CF` class: Cloudflare DNS records and tunnels through the official Python SDK

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from cloudflare import Cloudflare
from fastcore.all import L, first, Path

## CF

`CF` holds one connection to the Cloudflare API and offers a method for each job. The token comes from the `token` argument, or from the `CLOUDFLARE_API_TOKEN` environment variable when you leave it out. `timeout` is the number of seconds to wait for each request, and any other keyword arguments go straight to the SDK's `Cloudflare` client.

Every method returns plain dictionaries and lists, converted from the SDK's objects, so the results print and serialise without extra work.

The methods come in three groups:

- Zones and DNS records: `zones`, `zone_id`, `dns_records`, `create_record`, `delete_record` and `upsert_record`. A zone is a domain you manage on Cloudflare. The low-level methods take a zone ID; `upsert_record` takes the domain name and looks the ID up itself, so it is the one to use in most code.
- Tunnels: `account_id`, `tunnels`, `tunnel_id`, `create_tunnel`, `delete_tunnel`, `tunnel_token` and `tunnel_cname`. Tunnels belong to an account rather than a zone, so `tunnels`, `tunnel_id`, `create_tunnel`, `delete_tunnel` and `tunnel_token` take an optional `account_id` and use your first account when you leave it out. `tunnel_cname` writes the DNS record that points a name at a tunnel.
- Shortcuts: `verify` checks the token without changing anything, and `setup_tunnel` creates a tunnel, its DNS record and its token in one call.

`upsert_record` keeps one record per name. It walks the records that already have the name. The first one with the same type and content is kept and returned, and any record it passed on the way is deleted. If none match, every record under the name is deleted, whatever its type, and the new record is created. A name without a dot is treated as a subdomain of the domain.

In [ ]:
#| export
class CF:
    'A Cloudflare API client for DNS records and tunnels, built on the official `cloudflare` SDK.'
    def __init__(self, token=None, timeout=10, **kw):
        self._c = Cloudflare(api_token=token, timeout=timeout, **kw)

    def zones(self) -> list:
        'List every zone (domain) the token can see.'
        return [z.model_dump() for z in self._c.zones.list()]

    def zone_id(self, domain) -> str:
        'Return the zone ID of `domain`, raising `ValueError` if the token cannot see it.'
        for z in self._c.zones.list(name=domain): return z.id
        raise ValueError(f'Zone not found: {domain}')

    def dns_records(self, zone_id) -> list:
        'List the DNS records in the zone with ID `zone_id`.'
        return [r.model_dump() for r in self._c.dns.records.list(zone_id=zone_id)]

    def create_record(self, zone_id, type, name, content, ttl=1, proxied=False) -> dict:
        'Create a DNS record in a zone; `ttl=1` lets Cloudflare choose the time to live.'
        r = self._c.dns.records.create(zone_id=zone_id, type=type, name=name, content=content, ttl=ttl, proxied=proxied)
        return r.model_dump()

    def delete_record(self, zone_id, record_id) -> None:
        'Delete the DNS record `record_id` from the zone `zone_id`.'
        self._c.dns.records.delete(record_id, zone_id=zone_id)

    def upsert_record(self, domain, name, content, type='A', proxied=False) -> dict:
        'Make `name` in `domain` point at `content`, keeping a matching record and replacing any others under that name.'
        zid = self.zone_id(domain)
        f = name if '.' in name else f'{name}.{domain}'
        existing = list(self._c.dns.records.list(zone_id=zid, name=f))
        for r in existing:
            if r.type == type and r.content == content: return r.model_dump()
            self.delete_record(zid, r.id)
        return self.create_record(zid, type, name, content, proxied=proxied)

    def account_id(self) -> str:
        'Return the ID of the first account the token can see.'
        for a in self._c.accounts.list(): return a.id
        raise ValueError('No accounts found')

    def tunnels(self, account_id=None) -> list:
        'List the tunnels in an account, by default the first one.'
        aid = account_id or self.account_id()
        return [t.model_dump() for t in self._c.zero_trust.tunnels.cloudflared.list(account_id=aid)]

    def tunnel_id(self, name, account_id=None) -> str:
        'Return the ID of the tunnel called `name`, raising `ValueError` if there is none.'
        t = first(L(self.tunnels(account_id=account_id)).filter(lambda x: x.get('name') == name))
        if t is None: raise ValueError(f'Tunnel not found: {name}')
        return t['id']

    def create_tunnel(self, name, account_id=None) -> dict:
        'Create a tunnel called `name` and return it.'
        aid = account_id or self.account_id()
        t = self._c.zero_trust.tunnels.cloudflared.create(account_id=aid, name=name)
        return t.model_dump()

    def delete_tunnel(self, tunnel_id, account_id=None) -> None:
        'Delete the tunnel with ID `tunnel_id`.'
        aid = account_id or self.account_id()
        self._c.zero_trust.tunnels.cloudflared.delete(tunnel_id, account_id=aid)

    def tunnel_token(self, tunnel_id, account_id=None) -> str:
        'Return the secret token that `cloudflared` needs to connect to the tunnel.'
        aid = account_id or self.account_id()
        result = self._c.zero_trust.tunnels.cloudflared.token.get(tunnel_id, account_id=aid)
        return result if isinstance(result, str) else result.token

    def tunnel_cname(self, domain: str, name: str, tunnel_id: str, proxied: bool = True) -> dict:
	    'Point `name` in `domain` at a tunnel with a CNAME record to `<tunnel_id>.cfargotunnel.com`.'
	    return self.upsert_record(domain, name, f"{tunnel_id}.cfargotunnel.com",
	                              type='CNAME', proxied=proxied)

    def verify(self) -> dict:
        'Check, without changing anything, that the token can reach the account, its zones and its tunnels.'
        try:
	        acc = self.account_id()
	        return dict(result=bool(self._c.zones.list() and self._c.zero_trust.tunnels.cloudflared.list(account_id=acc)))
        except Exception as e: return dict(result=False, err=str(e))

    def setup_tunnel(self, domain: str, name: str | None = None, tunnel_name: str | None = None, proxied: bool = True):
        'Create or reuse a tunnel, point `name` (or the apex domain) at it, and return `(tunnel_id, token)`.'
        n = name or domain
        tname = tunnel_name or n
        exists = L.filter(lambda x: x.get('name') == tname)
        tid = first(t)['id'] if (t:=exists(self.tunnels())) else self.create_tunnel(tname)['id']
        self.tunnel_cname(domain, n, tid, proxied=proxied)
        return tid, self.tunnel_token(tid)

### Least-privilege token setup

Give the token only the permissions cfeasy needs. Create a custom token in the Cloudflare dashboard under My Profile -> API Tokens -> Create Token:

| What it is for | Permission |
|---|---|
| DNS records | Zone -> DNS -> Edit (limit it to your zone) |
| Finding the account ID for tunnel calls | Account -> Account Settings -> Read |
| Creating, listing and deleting tunnels | Account -> Cloudflare Tunnel -> Edit |

`verify` confirms the token works before you change anything. It returns `{'result': True}`, or `{'result': False, 'err': ...}` with the error message. The examples below ran against a real account and are not run by the docs build. The second `setup_tunnel` call serves the domain `sankalpa.sh` itself instead of a subdomain.

In [ ]:
#| eval: false
c = CF()
c.verify()

{'result': True}

In [ ]:
#| eval: false
# setup_tunnel: subdomain
tid, token = c.setup_tunnel('sankalpa.sh', 'app')   # → app.sankalpa.sh
tid, token

('b59c4690-7c89-4ab7-9421-176afa6e2dfa',
 'eyJhIjoiOTg3ODZiMGRlMTM5OTE0YmVkNGMyYjdmNGYwYzg4YjkiLCJ0IjoiYjU5YzQ2OTAtN2M4OS00YWI3LTk0MjEtMTc2YWZhNmUyZGZhIiwicyI6ImVJN1RyWE1RK1BvU09nQUp2UG5uWHdzT0h0TFRlU1FMd1ZwU1lXSW1PVXhJWWxseWhnL21DeW1DRTJialRIcEM1QW90aGRkaXZzc3VCQ0RUSEdVZ3dnPT0ifQ==')

In [ ]:
#| eval: false
# setup_tunnel: apex domain (no name → CNAME-flattened at root)
tid_apex, token_apex = c.setup_tunnel('sankalpa.sh')   # → example.com
tid_apex, token_apex

('6791fbaf-e848-422b-8fc7-35afc3e435d0',
 'eyJhIjoiOTg3ODZiMGRlMTM5OTE0YmVkNGMyYjdmNGYwYzg4YjkiLCJ0IjoiNjc5MWZiYWYtZTg0OC00MjJiLThmYzctMzVhZmMzZTQzNWQwIiwicyI6IjNNOVhaQm1IWkJPdDl3a28rRUxza3dmaWZ3ZUpqcmpSV0lxdDNRcExwTHFEZVJxaFRyeGRWZnR3M082TzJ1cFNOc0FYSXNRMGx0UjdhaE1SbVFxNDNnPT0ifQ==')

In [ ]:
#| eval: false
# tunnel_id: look up by name; delete_tunnel: clean up by name
c.tunnel_id('app')                          # → UUID
c.delete_tunnel(c.tunnel_id('app'))
c.delete_tunnel(c.tunnel_id('sankalpa.sh'))

## Helpers

`repo_root` returns the top folder of the git repository you are working in, found by walking up from the current directory until it finds a `.git` entry. It returns None outside a repository.

cfeasy ships a `SKILL.md` file that tells coding agents such as Claude Code how to use the library. `mv_skill_md` copies it into a project's `.agents/skills/cfeasy/` and `.claude/skills/cfeasy/` folders, where those agents look for skills. By default it only prints where it would copy the file; pass `dry_run=False` to copy it. The project root is `dir` if you give one, otherwise the repository root, otherwise the current directory.

In [ ]:
#| export
def repo_root() -> Path:
    'Find the root of the current git repository, or None if not in a repo.'
    return first((Path.cwd(), *Path.cwd().parents), lambda p: (p/'.git').exists())

def mv_skill_md(dry_run=True, dir=None) -> None:
    'Copy the bundled SKILL.md into the project\'s agent skill folders, or with `dry_run` only print where it would go.'
    base = Path(__file__).parent if '__file__' in globals() else Path.cwd()
    if not (src := base/'SKILL.md').exists(): return
    root = dir or repo_root() or '.'
    ts = [Path(root)/'.agents/skills/cfeasy/SKILL.md', Path(root)/'.claude/skills/cfeasy/SKILL.md']
    if dry_run: print(f'Would copy {src} to: {list(map(str,ts))}')
    else:
        for t in ts: t.mk_write(src.read_text(encoding='utf-8'))
        print(f'Installed -> {list(map(str,ts))}')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()